[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/danpele/SFM/blob/main/Quantlets/Ch_10/SFM_ch10_backtesting/SFM_ch10_backtesting.ipynb)

# SFM_ch10_backtesting

Backtesting VaR 1% forecasts of four methods (historical simulation, Normal, GARCH-t, FHS) for the S&P 500, DAX, BET (since 2005) and Bitcoin (since 2017): exceedance counts, the Kupiec (1995) proportion-of-failures test, the Christoffersen (1998) independence and conditional coverage tests, the Basel traffic light (Binomial(250, 1%) zones and plus factors of 1996) on rolling 250-day windows, the average quantile loss, the 2008 and 2020 stress periods, and the Acerbi-Szekely (2014) Z2 test of ES 2.5% with simulated p-values (as in SFEvar_pot_backtesting).

*Statistics of Financial Markets (SFM), Chapter 10: VaR, ES and backtesting. Author: Daniel Traian Pele.*

> **Working in Google Colab? Save your own copy first.**
> The Colab link opens a temporary copy: when you close the tab or the session times out, your changes are lost.
> Use **File → Save a copy in Drive** (it goes to the *Colab Notebooks* folder of your ASE Google Drive) and work in that copy.
> For the team project, **File → Save a copy in GitHub** puts the notebook directly in your team's repository.
> Files you create while the notebook runs (CSV, charts) are also temporary: set `SAVE_TO_DRIVE = True` in the next cell to keep them.

In [ ]:
# Optional (Colab only): keep the files you create in Google Drive
SAVE_TO_DRIVE = False   # set to True, run this cell and allow access to your Drive

import os
OUT_DIR = '.'
if SAVE_TO_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    OUT_DIR = '/content/drive/MyDrive/SFM/Chapter_10'
    os.makedirs(OUT_DIR, exist_ok=True)
print('Save your outputs to:', os.path.abspath(OUT_DIR))


In [ ]:
# the arch package (maximum-likelihood estimation of GARCH models) is not part of Google Colab
!pip install -q arch

In [ ]:
import os
import re
import json
import urllib.request
import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.colors import to_rgb
from scipy import optimize
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

## Chart style

In [ ]:
# Palette (RGB values of latex/preamble.tex)
MainBlue = '#1A3A6E'
IDAred = '#CD0000'
Forest = '#2E7D32'
Amber = '#B5853F'
Orange = '#E67E22'
Purple = '#8E44AD'
Teal = '#17A2B8'
Crimson = '#DC3545'
DarkText = '#1F2A44'          # text, axes and ticks (dark navy, not grey)

PALETTE = [MainBlue, IDAred, Forest, Amber, Purple, Orange, Teal, Crimson]
# fixed colours for the series used in several chapters
COL = {'sp500': MainBlue, 'bet': IDAred, 'bettr': Orange, 'dax': Teal, 'btc': Amber, 'eth': Crimson,
       'eurron': Forest, 'gold': Purple, 'vix': Crimson, 'stoxx50': Forest, 'ndx': Teal}


def apply():
    """Set the course style for matplotlib."""
    rc = plt.rcParams
    rc['figure.facecolor'] = 'none'
    rc['axes.facecolor'] = 'none'
    rc['savefig.facecolor'] = 'none'
    rc['savefig.transparent'] = True
    rc['axes.grid'] = False
    rc['font.family'] = 'sans-serif'
    rc['font.sans-serif'] = ['Helvetica', 'Arial', 'DejaVu Sans']
    # font sizes for slides (charts are 9-11 inches wide and shown at 8-14 cm)
    rc['font.size'] = 12
    rc['axes.labelsize'] = 13
    rc['axes.titlesize'] = 13
    rc['xtick.labelsize'] = 11.5
    rc['ytick.labelsize'] = 11.5
    rc['legend.fontsize'] = 11
    rc['axes.spines.top'] = False
    rc['axes.spines.right'] = False
    rc['axes.linewidth'] = 0.6
    rc['lines.linewidth'] = 1.4
    rc['legend.facecolor'] = 'none'
    rc['legend.framealpha'] = 0
    rc['legend.frameon'] = False
    for k in ('text.color', 'axes.labelcolor', 'axes.edgecolor', 'xtick.color', 'ytick.color', 'axes.titlecolor'):
        rc[k] = DarkText
    rc['axes.prop_cycle'] = mpl.cycler(color=PALETTE)


def legend_outside_bottom(ax, ncol=2, y=-0.22, **kw):
    """Place the legend outside the plot, bottom centre (for a figure with several axes, pass the last one
    or use fig.legend with the same arguments)."""
    return ax.legend(loc='upper center', bbox_to_anchor=(0.5, y), ncol=ncol, frameon=False, **kw)


def fig_legend_bottom(fig, handles=None, labels=None, ncol=3, y=-0.02):
    """One legend for a whole figure (several panels), below the panels."""
    if handles is None:
        handles, labels = [], []
        for ax in fig.axes:
            for h, l in zip(*ax.get_legend_handles_labels()):
                if l not in labels and not l.startswith('_'):
                    handles.append(h)
                    labels.append(l)
    return fig.legend(handles, labels, loc='upper center', bbox_to_anchor=(0.5, y), ncol=ncol, frameon=False)


def _is_grey(c, tol=0.06):
    try:
        r, g, b = to_rgb(c)
    except ValueError:
        return False
    return max(r, g, b) - min(r, g, b) < tol and 0.25 < (r + g + b) / 3 < 0.95


def check_no_grey(fig):
    """House rule: no grey series and no grey text. Raises ValueError listing the offending elements."""
    bad = []
    for ax in fig.axes:
        for ln in ax.get_lines():
            if not ln.get_label().startswith('_') and _is_grey(ln.get_color()):
                bad.append(f'line {ln.get_label()!r}')
        for coll in ax.collections:
            fc = coll.get_facecolor()
            if len(fc) and coll.get_label() and not coll.get_label().startswith('_') and _is_grey(fc[0][:3]):
                bad.append(f'series {coll.get_label()!r}')
        for t in ax.texts + [ax.title, ax.xaxis.label, ax.yaxis.label]:
            if t.get_text() and _is_grey(t.get_color()):
                bad.append(f'text {t.get_text()[:30]!r}')
    if bad:
        raise ValueError('grey elements: ' + ', '.join(bad))


def save_fig(name, out_dir=None, show=True):
    """Save the figure as transparent PDF and PNG (OUT_DIR if defined, else the current folder), then show it."""
    d = out_dir or globals().get('OUT_DIR', '.')
    plt.savefig(os.path.join(d, f'{name}.pdf'), bbox_inches='tight', transparent=True)
    plt.savefig(os.path.join(d, f'{name}.png'), bbox_inches='tight', transparent=True, dpi=180)
    if show:
        plt.show()
    plt.close()


apply()
# the chapter scripts call the style as st.<name> (import sfm_style as st): the same names here
import types
st = types.SimpleNamespace(COL=COL, PALETTE=PALETTE, MainBlue=MainBlue, IDAred=IDAred, Forest=Forest, Amber=Amber,
                           Orange=Orange, Purple=Purple, Teal=Teal, Crimson=Crimson, DarkText=DarkText, apply=apply, legend_outside_bottom=legend_outside_bottom,
                           fig_legend_bottom=fig_legend_bottom, save_fig=save_fig, check_no_grey=check_no_grey)

## Data

In [ ]:
REPO_RAW = 'https://raw.githubusercontent.com/danpele/SFM/main/data/market/'
# local copy of the course data, if the notebook runs inside the repository
MARKET_DIR = next((os.path.join(d, 'data', 'market') for d in ('.', '..', '../..', '../../..')
                   if os.path.isdir(os.path.join(d, 'data', 'market'))), '')
END = '2026-09-18'          # last day of the saved data

# name -> (symbol, label, group, field, default start)
SERIES = {
    # equity indices
    'sp500':    ('GSPC.INDX',     'S&P 500',               'Equity', 'close', '2000-01-01'),
    'ndx':      ('NDX.INDX',      'Nasdaq 100',            'Equity', 'close', '2000-01-01'),
    'dax':      ('GDAXI.INDX',    'DAX',                   'Equity', 'close', '2000-01-01'),
    'stoxx50':  ('STOXX50E.INDX', 'Euro Stoxx 50',         'Equity', 'close', '2000-01-01'),
    'nikkei':   ('N225.INDX',     'Nikkei 225',            'Equity', 'close', '2000-01-01'),
    'bet':      ('BET',           'BET',                   'Equity', 'close', '2000-01-01'),
    'bettr':    ('BETTR.INDX',    'BET-TR',                'Equity', 'close', '2014-09-23'),
    'betxt':    ('BETXT.INDX',    'BET-XT',                'Equity', 'close', '2011-08-12'),
    'betfi':    ('BETFI.INDX',    'BET-FI',                'Equity', 'close', '2012-01-25'),
    'wig20':    ('WIG20.INDX',    'WIG20',                 'Equity', 'close', '2000-01-01'),
    'bux':      ('BUX.INDX',      'BUX',                   'Equity', 'close', '2000-01-01'),
    'px':       ('PX.INDX',       'PX',                    'Equity', 'close', '2000-01-01'),
    'vix':      ('VIX.INDX',      'VIX',                   'Volatility', 'close', '2000-01-01'),
    # ETFs (adjusted close)
    'spy':      ('SPY.US',        'SPY',                   'ETF', 'adjusted_close', '2000-01-01'),
    'tlt':      ('TLT.US',        'TLT (US Treasuries 20y+)', 'ETF', 'adjusted_close', '2002-07-30'),
    'gld':      ('GLD.US',        'GLD (gold ETF)',        'ETF', 'adjusted_close', '2004-11-18'),
    'tvbetetf': ('TVBETETF.RO',   'TVBETETF',              'ETF', 'adjusted_close', '2015-06-12'),
    # Bucharest Stock Exchange (adjusted close)
    'tlv':      ('TLV.RO',        'Banca Transilvania',    'Stock', 'adjusted_close', '2000-01-01'),
    'snp':      ('SNP.RO',        'OMV Petrom',            'Stock', 'adjusted_close', '2001-09-04'),
    'brd':      ('BRD.RO',        'BRD',                   'Stock', 'adjusted_close', '2001-01-16'),
    'tgn':      ('TGN.RO',        'Transgaz',              'Stock', 'adjusted_close', '2008-01-25'),
    'sng':      ('SNG.RO',        'Romgaz',                'Stock', 'adjusted_close', '2013-11-12'),
    'snn':      ('SNN.RO',        'Nuclearelectrica',      'Stock', 'adjusted_close', '2013-11-04'),
    'h2o':      ('H2O.RO',        'Hidroelectrica',        'Stock', 'adjusted_close', '2023-07-12'),
    # US stocks (adjusted close)
    'aapl':     ('AAPL.US',       'Apple',                 'Stock', 'adjusted_close', '2000-01-01'),
    'msft':     ('MSFT.US',       'Microsoft',             'Stock', 'adjusted_close', '2000-01-01'),
    'nvda':     ('NVDA.US',       'NVIDIA',                'Stock', 'adjusted_close', '2000-01-01'),
    'jpm':      ('JPM.US',        'JPMorgan Chase',        'Stock', 'adjusted_close', '2000-01-01'),
    # crypto (close, 7 days a week)
    'btc':      ('BTC-USD.CC',    'Bitcoin',               'Crypto', 'close', '2014-09-17'),
    'eth':      ('ETH-USD.CC',    'Ethereum',              'Crypto', 'close', '2015-08-07'),
    'sol':      ('SOL-USD.CC',    'Solana',                'Crypto', 'close', '2020-04-11'),
    'usdt':     ('USDT-USD.CC',   'Tether (USDT)',         'Crypto', 'close', '2015-02-26'),
    # FX and commodities
    'eurron':   ('REF:EUR',       'EUR/RON (BNR reference)', 'FX', 'close', '2005-07-01'),
    'usdron':   ('REF:USD',       'USD/RON (BNR reference)', 'FX', 'close', '2005-07-01'),
    'eurron_eodhd': ('EURRON.FOREX', 'EUR/RON (EODHD)',    'FX', 'close', '2005-07-01'),
    'eurusd':   ('EURUSD.FOREX',  'EUR/USD',               'FX', 'close', '2002-05-06'),
    'gold':     ('XAUUSD.FOREX',  'Gold (XAU/USD)',        'Commodity', 'close', '2000-01-01'),
    # government bond yields (close, % p.a.)
    'us10y':    ('US10Y.GBOND',   'US 10Y yield',          'Yield', 'close', '2000-01-01'),
    'de10y':    ('DE10Y.GBOND',   'Germany 10Y yield',     'Yield', 'close', '2007-03-28'),
    'ro10y':    ('RO10Y.GBOND',   'Romania 10Y yield',     'Yield', 'close', '2007-08-17'),
}
LABELS = {k: v[1] for k, v in SERIES.items()}
_CACHE = {}


def read_market(symbol):
    """Daily table of one symbol: local copy of the course data, otherwise the SFM repository on GitHub."""
    fname = f'{symbol}.csv'
    path = os.path.join(MARKET_DIR, fname) if MARKET_DIR else ''
    src = path if path and os.path.exists(path) else REPO_RAW + fname
    return pd.read_csv(src, index_col='date', parse_dates=True).sort_index()


def read_reference_rate(currency='EUR', start='2005-07-01', end=END):
    """Official BNR reference rate (RON per unit of currency), from the public yearly XML archives."""
    key = (currency, start, end)
    if key in _CACHE:
        return _CACHE[key]
    rows = []
    for y in range(max(int(start[:4]), 2005), int(end[:4]) + 1):
        url = f'https://curs.bnr.ro/files/xml/years/nbrfxrates{y}.xml'
        xml = urllib.request.urlopen(urllib.request.Request(url, headers={'User-Agent': 'Mozilla'}),
                                     timeout=60).read().decode()
        for d, body in re.findall(r'<Cube date="([\d-]+)">(.*?)</Cube>', xml, re.S):
            m = re.search(rf'<Rate currency="{currency}"(?: multiplier="\d+")?>([\d.]+)</Rate>', body)
            if m:
                rows.append((d, float(m.group(1))))
    s = pd.DataFrame(rows, columns=['date', 'close']).drop_duplicates('date').set_index('date')['close']
    s.index = pd.to_datetime(s.index)
    _CACHE[key] = s.sort_index().loc[start:end]
    return _CACHE[key]


def load_close(name, start=None, end=END, field=None):
    """Daily closing price of a named series, cleaned with the course conventions."""
    symbol, label, group, field0, start0 = SERIES[name]
    start, field = start or start0, field or field0
    if symbol.startswith('REF:'):
        return read_reference_rate(symbol.split(':')[1], start=start, end=end).rename(name)
    t = read_market(symbol)
    s = (t[field] if field in t.columns else t['close']).loc[start:end]
    s = pd.to_numeric(s, errors='coerce').dropna()
    if group != 'Yield':
        s = s[s > 0]
    if group != 'Crypto':
        s = s[s.index.dayofweek < 5]          # no weekend quotes
        s = s[s.diff() != 0]                  # no holidays filled with the previous price
    s.index.name = 'date'
    return s.rename(name)


def load_ohlc(name, start=None, end=END):
    """Open, high, low, close (and volume) of a series with OHLC data (range-based volatility estimators)."""
    symbol, _, group, _, start0 = SERIES[name]
    t = read_market(symbol).loc[start or start0:end]
    cols = [c for c in ('open', 'high', 'low', 'close', 'volume') if c in t.columns]
    t = t[cols].dropna()
    if group != 'Crypto':
        t = t[t.index.dayofweek < 5]
    return t[(t[['open', 'high', 'low', 'close']] > 0).all(axis=1)] if 'open' in cols else t


def log_returns(name, start=None, end=END, pct=True):
    """Daily log returns r_t = ln P_t - ln P_{t-1} (in % by default), on the series' own calendar."""
    r = np.log(load_close(name, start, end)).diff().dropna()
    return (100 * r if pct else r).rename(name)


def simple_returns(name, start=None, end=END, pct=True):
    """Daily simple returns R_t = P_t / P_{t-1} - 1 (in % by default)."""
    r = load_close(name, start, end).pct_change().dropna()
    return (100 * r if pct else r).rename(name)


def load_panel(names, start=None, end=END, kind='close'):
    """Several series aligned on date: kind = 'close' or 'returns' (NaN where a market does not trade)."""
    f = load_close if kind == 'close' else log_returns
    return pd.concat([f(n, start, end) for n in names], axis=1)


def periods_per_year(r):
    """Average number of observations per calendar year (the actual frequency of the series)."""
    years = (r.index[-1] - r.index[0]).days / 365.25
    return len(r) / years

## Analysis

In [ ]:
from arch import arch_model

NAME = {'sp500': 'S&P 500', 'dax': 'DAX', 'bet': 'BET', 'btc': 'Bitcoin', 'tlv': 'Banca Transilvania', 'snp': 'OMV Petrom', 'brd': 'BRD'}

START = {'sp500': '2000-01-01', 'dax': '2000-01-01', 'bet': '2000-01-01', 'btc': None, 'tlv': '2010-01-01', 'snp': '2010-01-01', 'brd': '2010-01-01'}

ASSETS = ['sp500', 'dax', 'bet', 'btc', 'tlv', 'snp']

BT_ASSETS = ['sp500', 'dax', 'bet', 'btc']

BAD_DAYS = {'tlv': ['2016-05-30', '2016-05-31']}

COLORS = {'sp500': '#1A3A6E', 'dax': '#17A2B8', 'bet': '#CD0000', 'btc': '#B5853F', 'tlv': '#E67E22', 'snp': '#2E7D32'}

EPISODES = {'2008': ('2008-09-01', '2009-03-31'), '2020': ('2020-02-15', '2020-05-31')}

ALPHA_VAR = 0.01

ALPHA_ES = 0.025

WINDOW = 500

REFIT = 250

OOS_START = {'sp500': '2005-01-01', 'dax': '2005-01-01', 'bet': '2005-01-01', 'btc': '2017-01-01'}

POT_Q = 0.9

METHODS = ['HS', 'Normal', 'GARCH-t', 'FHS']

MCOL = {'HS': '#B5853F', 'Normal': '#2E7D32', 'GARCH-t': '#CD0000', 'FHS': '#1A3A6E'}

PORT = ('bet', 'sp500')

WEIGHTS = (0.5, 0.5)

N_SIM = 200000

N_Z2 = 1000

SEED = 2026

PLUS = {0: 0.0, 1: 0.0, 2: 0.0, 3: 0.0, 4: 0.0, 5: 0.4, 6: 0.5, 7: 0.65, 8: 0.75, 9: 0.85}

def returns(k, start=None, end=None):
    """Daily log returns in % on the series' own calendar; known data errors removed."""
    r = log_returns(k, start or START.get(k)) if end is None else log_returns(k, start or START.get(k), end)
    if k in BAD_DAYS:
        r = r.drop(pd.to_datetime(BAD_DAYS[k]), errors='ignore')
    return r


def hs_var(x, a=ALPHA_VAR):
    """Historical simulation: VaR_a = -r_(k), the k-th smallest return, k = ceil(n a)."""
    xs = np.sort(np.asarray(x, float))
    k = int(np.ceil(len(xs) * a - 1e-9))
    return float(-xs[k - 1])


def hs_es(x, a=ALPHA_ES):
    """Historical simulation: ES_a = minus the average of the k smallest returns, k = ceil(n a)."""
    xs = np.sort(np.asarray(x, float))
    k = int(np.ceil(len(xs) * a - 1e-9))
    return float(-xs[:k].mean())


def normal_var(mu, sd, a=ALPHA_VAR):
    """VaR_a = -(mu + sd z_a), z_a the a-quantile of N(0, 1) (negative)."""
    return float(-(mu + sd * stats.norm.ppf(a)))


def normal_es(mu, sd, a=ALPHA_ES):
    """ES_a = -mu + sd phi(z_a) / a."""
    return float(-mu + sd * stats.norm.pdf(stats.norm.ppf(a)) / a)


def t_es_factor(nu, a=ALPHA_ES):
    """ES of a Student-t variable with nu degrees of freedom (scale 1): f(t_a) (nu + t_a^2) / ((nu - 1) a)."""
    q = stats.t.ppf(a, nu)
    return float(stats.t.pdf(q, nu) * (nu + q ** 2) / ((nu - 1) * a))


def std_t_q(nu, a=ALPHA_VAR):
    """a-quantile of the standardised Student-t (variance 1): t_nu^{-1}(a) sqrt((nu - 2)/nu)."""
    return float(stats.t.ppf(a, nu) * np.sqrt((nu - 2) / nu))


def std_t_es(nu, a=ALPHA_ES):
    """ES_a of the standardised Student-t (variance 1)."""
    return float(np.sqrt((nu - 2) / nu) * t_es_factor(nu, a))


def t_fit(x):
    """Student-t with location m, scale s and nu degrees of freedom, fitted by maximum likelihood (scipy)."""
    nu, m, s = stats.t.fit(np.asarray(x, float))
    return float(nu), float(m), float(s)


def t_var_es(x, a_var=ALPHA_VAR, a_es=ALPHA_ES):
    nu, m, s = t_fit(x)
    return {'var': float(-(m + s * stats.t.ppf(a_var, nu))), 'es': float(-m + s * t_es_factor(nu, a_es)),
            'nu': nu, 'm': m, 's': s}


def cf_z(a, S, K):
    """Cornish-Fisher quantile of a standardised variable with skewness S and excess kurtosis K."""
    z = stats.norm.ppf(a)
    return z + (z ** 2 - 1) * S / 6 + (z ** 3 - 3 * z) * K / 24 - (2 * z ** 3 - 5 * z) * S ** 2 / 36


def cf_var_es(x, a_var=ALPHA_VAR, a_es=ALPHA_ES):
    """Cornish-Fisher VaR = -(mu + sd z_cf); ES = minus the average of the CF quantiles on (0, a_es)."""
    x = np.asarray(x, float)
    mu, sd = x.mean(), x.std(ddof=1)
    S, K = float(stats.skew(x)), float(stats.kurtosis(x))
    u = (np.arange(2000) + 0.5) / 2000 * a_es
    return {'var': float(-(mu + sd * cf_z(a_var, S, K))), 'es': float(-(mu + sd * cf_z(u, S, K).mean())),
            'z': float(cf_z(a_var, S, K)), 'S': S, 'K': K, 'mu': float(mu), 'sd': float(sd)}


def pot_fit(x, q=POT_Q):
    """GPD fit to the losses L = -r above the threshold u = q-quantile of the losses (peaks over threshold)."""
    L = -np.asarray(x, float)
    u = float(np.quantile(L, q))
    exc = L[L > u] - u
    xi, _, beta = stats.genpareto.fit(exc, floc=0)
    return {'u': u, 'xi': float(xi), 'beta': float(beta), 'Nu': int(len(exc)), 'n': int(len(L))}


def pot_var(f, a):
    """VaR_a = u + beta/xi [(n a / N_u)^(-xi) - 1], valid for a < N_u / n."""
    return float(f['u'] + f['beta'] / f['xi'] * ((f['n'] * a / f['Nu']) ** (-f['xi']) - 1))


def pot_es(f, a):
    """ES_a = VaR_a / (1 - xi) + (beta - xi u) / (1 - xi), valid for xi < 1."""
    return float(pot_var(f, a) / (1 - f['xi']) + (f['beta'] - f['xi'] * f['u']) / (1 - f['xi']))


def garch_fit(r, last_obs=None):
    """GARCH(1,1) with constant mean and standardised Student-t innovations (arch package)."""
    am = arch_model(r, mean='Constant', vol='GARCH', p=1, q=1, dist='t')
    return am.fit(disp='off', last_obs=last_obs, options={'maxiter': 2000})


def rolling_forecasts(k, oos_start=None, window=WINDOW, refit=REFIT):
    """One-day-ahead VaR 1%, VaR 2.5% and ES 2.5% for every day from oos_start; the forecast for day t uses only data
    up to day t-1. HS and Normal: the last `window` returns. GARCH-t and FHS: GARCH(1,1)-t re-estimated every `refit`
    days on all data up to that day; FHS uses the empirical quantiles of the standardised residuals."""
    r = returns(k)
    x = r.values
    i0 = max(r.index.searchsorted(pd.Timestamp(oos_start or OOS_START[k])), window)
    W = np.lib.stride_tricks.sliding_window_view(x, window)[i0 - window:len(x) - window]    # windows ending at t-1
    Ws = np.sort(W, axis=1)
    k1 = int(np.ceil(window * ALPHA_VAR - 1e-9))
    k25 = int(np.ceil(window * ALPHA_ES - 1e-9))
    df = pd.DataFrame(index=r.index[i0:])
    df['r'] = x[i0:]
    df['var1_HS'] = -Ws[:, k1 - 1]
    df['var25_HS'] = -Ws[:, k25 - 1]
    df['es25_HS'] = -Ws[:, :k25].mean(axis=1)
    m, s = W.mean(axis=1), W.std(axis=1, ddof=1)
    df['mu_N'], df['sd_N'] = m, s
    df['var1_Normal'] = -(m + s * stats.norm.ppf(ALPHA_VAR))
    df['var25_Normal'] = -(m + s * stats.norm.ppf(ALPHA_ES))
    df['es25_Normal'] = -m + s * stats.norm.pdf(stats.norm.ppf(ALPHA_ES)) / ALPHA_ES
    cols = {c: [] for c in ['mu_G', 'sig_G', 'nu_G', 'block']}
    zs = []
    for b, s0 in enumerate(range(i0, len(r), refit)):
        e = min(s0 + refit, len(r))
        am = arch_model(r.iloc[:e], mean='Constant', vol='GARCH', p=1, q=1, dist='t')
        res = am.fit(disp='off', last_obs=r.index[s0], options={'maxiter': 2000})
        sig = am.fix(res.params).conditional_volatility.iloc[s0:e].values
        p = res.params
        cols['mu_G'] += [float(p['mu'])] * (e - s0)
        cols['sig_G'] += list(sig)
        cols['nu_G'] += [float(p['nu'])] * (e - s0)
        cols['block'] += [b] * (e - s0)
        zs.append(np.sort(res.std_resid.dropna().values))
    for c, v in cols.items():
        df[c] = v
    mu, sig, nu = df['mu_G'].values, df['sig_G'].values, df['nu_G'].values
    df['var1_GARCH-t'] = -(mu + sig * stats.t.ppf(ALPHA_VAR, nu) * np.sqrt((nu - 2) / nu))
    df['var25_GARCH-t'] = -(mu + sig * stats.t.ppf(ALPHA_ES, nu) * np.sqrt((nu - 2) / nu))
    qa = stats.t.ppf(ALPHA_ES, nu)
    df['es25_GARCH-t'] = -mu + sig * np.sqrt((nu - 2) / nu) * stats.t.pdf(qa, nu) * (nu + qa ** 2) / ((nu - 1) * ALPHA_ES)
    zq1 = np.array([-hs_var(z, ALPHA_VAR) for z in zs])[df['block'].values]
    zq25 = np.array([-hs_var(z, ALPHA_ES) for z in zs])[df['block'].values]
    zes = np.array([hs_es(z, ALPHA_ES) for z in zs])[df['block'].values]
    df['var1_FHS'] = -(mu + sig * zq1)
    df['var25_FHS'] = -(mu + sig * zq25)
    df['es25_FHS'] = -mu + sig * zes
    df.attrs['W'] = W
    df.attrs['zs'] = zs
    return df


def kupiec(x, n, a=ALPHA_VAR):
    """Kupiec (1995) proportion-of-failures test: LR_uc = -2 ln[(1-a)^(n-x) a^x / ((1-x/n)^(n-x) (x/n)^x)] ~ chi2(1)."""
    ph = x / n
    l0 = (n - x) * np.log(1 - a) + x * np.log(a)
    l1 = (n - x) * np.log(1 - ph) if ph < 1 else 0.0
    l1 += x * np.log(ph) if x > 0 else 0.0
    lr = float(-2 * (l0 - l1))
    return {'x': int(x), 'n': int(n), 'rate': float(ph), 'lr': lr, 'p': float(stats.chi2.sf(lr, 1))}


def christoffersen(hits, a=ALPHA_VAR):
    """Christoffersen (1998): independence LR_ind ~ chi2(1) from the transition counts n_ij (i = yesterday,
    j = today), and conditional coverage LR_cc = LR_uc + LR_ind ~ chi2(2)."""
    h = np.asarray(hits, int)
    h0, h1 = h[:-1], h[1:]
    n00 = int(np.sum((h0 == 0) & (h1 == 0)))
    n01 = int(np.sum((h0 == 0) & (h1 == 1)))
    n10 = int(np.sum((h0 == 1) & (h1 == 0)))
    n11 = int(np.sum((h0 == 1) & (h1 == 1)))
    p01 = n01 / (n00 + n01) if n00 + n01 else 0.0
    p11 = n11 / (n10 + n11) if n10 + n11 else 0.0
    p = (n01 + n11) / (n00 + n01 + n10 + n11)

    def ll(q, a_, b_):
        return (a_ * np.log(1 - q) if a_ and q < 1 else 0.0) + (b_ * np.log(q) if b_ and q > 0 else 0.0)
    lr_ind = float(-2 * (ll(p, n00 + n10, n01 + n11) - ll(p01, n00, n01) - ll(p11, n10, n11)))
    uc = kupiec(int(h.sum()), len(h), a)
    lr_cc = uc['lr'] + lr_ind
    return {'n00': n00, 'n01': n01, 'n10': n10, 'n11': n11, 'p01': float(p01), 'p11': float(p11),
            'lr_ind': lr_ind, 'p_ind': float(stats.chi2.sf(lr_ind, 1)), 'lr_cc': float(lr_cc),
            'p_cc': float(stats.chi2.sf(lr_cc, 2)), 'uc': uc}


def basel_zone(x):
    """Basel traffic light for 250 days at VaR 1%: green 0-4, yellow 5-9, red 10 or more exceptions."""
    return 'green' if x <= 4 else ('yellow' if x <= 9 else 'red')


def traffic_table(n=250, a=ALPHA_VAR, xmax=10):
    """Binomial(n, a) probability and cumulative probability of x exceptions, zone and plus factor (BCBS, 1996)."""
    return [{'x': x, 'p': float(stats.binom.pmf(x, n, a)), 'cum': float(stats.binom.cdf(x, n, a)), 'zone': basel_zone(x),
             'plus': PLUS.get(x, 1.0)} for x in range(xmax + 1)]


def z2_stat(r, var25, es25, a=ALPHA_ES):
    """Acerbi-Szekely (2014) Z2 = sum_t r_t I_t / (T a ES_t) + 1, I_t = 1{r_t < -VaR_t}; E[Z2] = 0 under the null,
    negative values: ES too small."""
    r, var25, es25 = (np.asarray(v, float) for v in (r, var25, es25))
    I = r < -var25
    return float(np.sum(r * I / es25) / (len(r) * a) + 1)


def z2_pvalue(df, m, nsim=N_Z2, seed=SEED):
    """p-value of Z2 by simulation: returns are drawn from each day's forecast distribution of method m and Z2 is
    recomputed with the same VaR and ES forecasts; p = share of simulated Z2 below the observed one."""
    rng = np.random.default_rng(seed)
    n = len(df)
    if m == 'Normal':
        X = df['mu_N'].values[:, None] + df['sd_N'].values[:, None] * rng.standard_normal((n, nsim))
    elif m == 'HS':
        W = df.attrs['W']
        X = W[np.arange(n)[:, None], rng.integers(0, W.shape[1], (n, nsim))]
    elif m == 'GARCH-t':
        nu = df['nu_G'].values[:, None]
        z = rng.standard_t(nu, (n, nsim)) * np.sqrt((nu - 2) / nu)
        X = df['mu_G'].values[:, None] + df['sig_G'].values[:, None] * z
    else:
        zs = df.attrs['zs']
        Z = np.empty((n, nsim))
        blk = df['block'].values
        for b, z in enumerate(zs):
            rows = np.where(blk == b)[0]
            Z[rows] = z[rng.integers(0, len(z), (len(rows), nsim))]
        X = df['mu_G'].values[:, None] + df['sig_G'].values[:, None] * Z
    v = df[f'var25_{m}'].values[:, None]
    e = df[f'es25_{m}'].values[:, None]
    zsim = np.sum(X * (X < -v) / e, axis=0) / (n * ALPHA_ES) + 1
    z0 = z2_stat(df['r'], df[f'var25_{m}'], df[f'es25_{m}'])
    return z0, float(np.mean(zsim <= z0))


def quantile_loss(r, var, a=ALPHA_VAR):
    """Average quantile (pinball) loss of VaR forecasts: S(v, x) = (a - 1{x < -v})(x + v); smaller is better."""
    r, var = np.asarray(r, float), np.asarray(var, float)
    return float(np.mean((a - (r < -var)) * (r + var)))


def backtest(df, m):
    """All backtests of method m on a frame of rolling forecasts."""
    hits = (df['r'] < -df[f'var1_{m}']).astype(int).values
    c = christoffersen(hits)
    last = hits[-250:].sum()
    z2, pz2 = z2_pvalue(df, m)
    return {'n': int(len(hits)), 'x': int(hits.sum()), 'rate': float(hits.mean()), 'exp': float(ALPHA_VAR * len(hits)),
            'lr_uc': c['uc']['lr'], 'p_uc': c['uc']['p'], 'lr_ind': c['lr_ind'], 'p_ind': c['p_ind'], 'lr_cc': c['lr_cc'],
            'p_cc': c['p_cc'], 'n11': c['n11'], 'n01': c['n01'], 'n10': c['n10'], 'n00': c['n00'], 'p01': c['p01'],
            'p11': c['p11'], 'last250': int(last), 'zone_last': basel_zone(last), 'z2': z2, 'p_z2': pz2,
            'zones': {z: int(np.sum([basel_zone(v) == z for v in pd.Series(hits).rolling(250).sum().dropna().values]))
                      for z in ('green', 'yellow', 'red')},
            'n_windows': int(len(hits) - 249), 'qloss': quantile_loss(df['r'], df[f'var1_{m}']),
            'max250': int(pd.Series(hits).rolling(250).sum().max())}


def stress_counts(df, m):
    """Exceedances of VaR 1% in the 2008 and 2020 stress periods."""
    out = {}
    for e, (a, b) in EPISODES.items():
        d = df.loc[a:b]
        out[e] = {'n': int(len(d)), 'x': int((d['r'] < -d[f'var1_{m}']).sum()), 'exp': float(ALPHA_VAR * len(d))}
    return out


def backtest_all(assets=BT_ASSETS):
    frames, res = {}, {}
    for k in assets:
        df = rolling_forecasts(k)
        frames[k] = df
        res[k] = {'start': df.index[0].date().isoformat(), 'end': df.index[-1].date().isoformat()}
        for m in METHODS:
            res[k][m] = backtest(df, m)
            res[k][m]['stress'] = stress_counts(df, m)
            res[k][m]['mean_var'] = float(df[f'var1_{m}'].mean())
    return res, frames


def fig_binomial(n=250, a=ALPHA_VAR, save=True):
    """Binomial(250, 1%) distribution of the number of exceptions and the Basel zones."""
    tt = traffic_table(n, a, 14)
    fig, ax = plt.subplots(figsize=(10, 3.8))
    col = {'green': st.Forest, 'yellow': st.Amber, 'red': st.IDAred}
    lab = {'green': 'green zone (0-4)', 'yellow': 'yellow zone (5-9)', 'red': 'red zone (10 or more)'}
    seen = set()
    for row in tt:
        z = row['zone']
        ax.bar(row['x'], 100 * row['p'], color=col[z], label=lab[z] if z not in seen else '_')
        seen.add(z)
        if row['x'] <= 9:
            ax.text(row['x'], 100 * row['p'] + 0.6, f"{100 * row['cum']:.2f}", ha='center', fontsize=9, color=st.DarkText)
    ax.set_xlabel('number of exceptions in 250 days (labels: cumulative probability, %)')
    ax.set_ylabel('probability (%)')
    ax.set_xticks(range(0, 15))
    st.legend_outside_bottom(ax, ncol=3, y=-0.25)
    st.check_no_grey(fig)
    if save:
        st.save_fig('sfm_ch10_binomial')
    return tt


def fig_hits(df, k='sp500', save=True):
    """Exceedance dates of the VaR 1% of each method: clusters show a lack of independence."""
    fig, ax = plt.subplots(figsize=(11, 3.6))
    for e, (a, b) in EPISODES.items():
        ax.axvspan(pd.Timestamp(a), pd.Timestamp(b), color=st.Amber, alpha=0.15, label='_')
    for i, m in enumerate(METHODS):
        hit = df.index[df['r'] < -df[f'var1_{m}']]
        ax.scatter(hit, np.full(len(hit), i), color=MCOL[m], marker='|', s=180, lw=1.6,
                   label=f'{m}: {len(hit)} exceedances')
    ax.set_yticks(range(len(METHODS)))
    ax.set_yticklabels(METHODS)
    ax.set_ylim(-0.6, len(METHODS) - 0.4)
    ax.set_xlabel('date (shaded: September 2008 - March 2009 and February - May 2020)')
    st.legend_outside_bottom(ax, ncol=4, y=-0.32)
    st.check_no_grey(fig)
    if save:
        st.save_fig('sfm_ch10_hits')


def fig_traffic_time(df, k='sp500', save=True):
    """Number of exceptions in the last 250 days for HS and GARCH-t, with the zone limits."""
    fig, ax = plt.subplots(figsize=(11, 3.8))
    for m in ('HS', 'Normal', 'GARCH-t'):
        c = (df['r'] < -df[f'var1_{m}']).astype(int).rolling(250).sum()
        ax.plot(c.index, c, color=MCOL[m], lw=1.5, label=f'{m}')
    ax.axhline(4.5, color=st.Forest, ls='--', lw=1.2, label='upper limit of the green zone (4)')
    ax.axhline(9.5, color=st.IDAred, ls='--', lw=1.2, label='upper limit of the yellow zone (9)')
    ax.set_ylabel('exceptions in 250 days')
    st.legend_outside_bottom(ax, ncol=3, y=-0.18)
    st.check_no_grey(fig)
    if save:
        st.save_fig('sfm_ch10_traffic_time')

## Run

In [ ]:
print(pd.DataFrame(fig_binomial()))
bt, frames = backtest_all()
print(pd.DataFrame({(NAME[k], m): {c: bt[k][m][c] for c in ['n', 'x', 'rate', 'p_uc', 'p_ind', 'p_cc', 'max250', 'qloss', 'z2', 'p_z2']} for k in BT_ASSETS for m in METHODS}).T.round(4))
fig_hits(frames['sp500'])
fig_traffic_time(frames['sp500'])

![sfm_ch10_binomial](./sfm_ch10_binomial.png)

Output: `sfm_ch10_binomial.pdf`

![sfm_ch10_hits](./sfm_ch10_hits.png)

Output: `sfm_ch10_hits.pdf`

![sfm_ch10_traffic_time](./sfm_ch10_traffic_time.png)

Output: `sfm_ch10_traffic_time.pdf`